In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
import torch

In [4]:
#Загрузка данных
df_train_origin = pd.merge(pd.read_csv('train.csv'), 
                           pd.read_csv('trainLabels.csv'), 
                           left_index=True,
                           right_index=True)
df_test_origin = pd.read_csv('test.csv')
df_full = pd.concat([df_train_origin, df_test_origin], axis=0)

In [5]:
#Корреляция признаков с целевой переменной
df_corr = df_train_origin.corr()['TARGET']

In [6]:
#Для обучения оставим столбцы с корреляцией по модулю больше 0.1
train_columns = df_corr.index[np.abs(df_corr) > 0.1].values
df_full = df_full[train_columns]

In [49]:
#Разбиение на training и result
df_training = df_full[df_full['TARGET'].notna()]
X_result = df_full[df_full['TARGET'].isna()].drop('TARGET', axis=1)

In [ ]:
#Обучение

In [39]:
#SVC
X = df_training.drop('TARGET', axis=1)
y = df_training['TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

scaler = StandardScaler()
model = SVC()
pipe_model = Pipeline([('scaler', scaler), ('model', model)])
grid_params = {'model__C': np.linspace(1,6,20)}
grid_model = GridSearchCV(pipe_model, grid_params, scoring='accuracy', cv=5, verbose=2)
grid_model.fit(X_train, y_train)
y_predict = grid_model.predict(X_test)
print(f'Точность на тесте = {accuracy_score(y_test, y_predict)}')
print(f'Лучшие параметры = {grid_model.best_params_}')

Fitting 5 folds for each of 20 candidates, totalling 100 fits
[CV] END .......................................model__C=1.0; total time=   0.0s
[CV] END .......................................model__C=1.0; total time=   0.0s
[CV] END .......................................model__C=1.0; total time=   0.0s
[CV] END .......................................model__C=1.0; total time=   0.0s
[CV] END .......................................model__C=1.0; total time=   0.0s
[CV] END .........................model__C=1.263157894736842; total time=   0.0s
[CV] END .........................model__C=1.263157894736842; total time=   0.0s
[CV] END .........................model__C=1.263157894736842; total time=   0.0s
[CV] END .........................model__C=1.263157894736842; total time=   0.0s
[CV] END .........................model__C=1.263157894736842; total time=   0.0s
[CV] END .........................model__C=1.526315789473684; total time=   0.0s
[CV] END .........................model__C=1.52

In [11]:
#XGBoost
X = df_training.drop('TARGET', axis=1)
y = df_training['TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

model = XGBClassifier()
pipe_model = Pipeline([('model', model)])
grid_params = {'model__n_estimators': list(range(10,300,30)),
               'model__max_depth': list(range(3,20,2))}
grid_model = GridSearchCV(pipe_model, grid_params, scoring='accuracy', cv=5, verbose=2)
grid_model.fit(X_train, y_train)
y_predict = grid_model.predict(X_test)
print(f'Точность на тесте = {accuracy_score(y_test, y_predict)}')
print(f'Лучшие параметры = {grid_model.best_params_}')

Fitting 5 folds for each of 90 candidates, totalling 450 fits
[CV] END .........model__max_depth=3, model__n_estimators=10; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=10; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=10; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=10; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=10; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=40; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=40; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=40; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=40; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=40; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_estimators=70; total time=   0.0s
[CV] END .........model__max_depth=3, model__n_

In [29]:
#Perceptron
class Perceptron(torch.nn.Module):
    def __init__(self,
                 in_features,
                 out_features,
                 hiden_dim,
                 num_layers):
        super(Perceptron, self).__init__()
        self.layers = torch.nn.Sequential()
        prev_features = in_features
        for layer_no in range(num_layers):
            self.layers.add_module(f'Linear{layer_no+1}',
                                   torch.nn.Linear(prev_features, hiden_dim))
            self.layers.add_module(f'ReLU{layer_no+1}',
                                   torch.nn.ReLU())
            prev_features = hiden_dim
        self.layers.add_module('regression',
                               torch.nn.Linear(prev_features, out_features))

    def forward(self, input):
        return self.layers(input)

def trainer(model, X, y, optimizer, loss_function, epochs):
    for epoch_no in range(epochs):
        optimizer.zero_grad()
        output = model(X)
        loss = loss_function(output, y)
        loss.backward()
        optimizer.step()    

In [38]:
#Подготовка данных
X = df_training.drop('TARGET', axis=1)
y = df_training['TARGET']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=101)

X_train_tensor = torch.from_numpy(X_train.astype(np.float32).to_numpy())
y_train_tensor = torch.from_numpy(y_train.astype(np.int64).to_numpy())
X_test_tensor = torch.from_numpy(X_test.astype(np.float32).to_numpy())
y_test_tensor = torch.from_numpy(y_test.astype(np.int64).to_numpy())

#Обучение нейронной сети
epochs = 1000
nn_model = Perceptron(in_features=X_train_tensor.shape[1], 
                      out_features=2, 
                      hiden_dim=11, 
                      num_layers=3)

_ = nn_model.train()
trainer(nn_model,  
        X_train_tensor,
        y_train_tensor,
        torch.optim.Adam(nn_model.parameters()),
        torch.nn.CrossEntropyLoss(),
        epochs)

with torch.no_grad():
    _ = nn_model.eval()
    y_predict = torch.argmax(nn_model(X_test_tensor), dim=-1)
    print(f'Ошибка на тесте = {accuracy_score(y_test_tensor, y_predict)}')

Ошибка на тесте = 0.89


In [51]:
#Будем использовать SVC
#SVC
X = df_training.drop('TARGET', axis=1)
y = df_training['TARGET']

scaler = StandardScaler()
model = SVC(C=4.1579)
final_model = Pipeline([('scaler', scaler), ('model', model)])
final_model.fit(X, y)
y_predict = final_model.predict(X)
print(f'Точность на тесте = {accuracy_score(y, y_predict)}')
print(f'Лучшие параметры = {final_model['model'].C}')

Точность на тесте = 0.962
Лучшие параметры = 4.1579


In [56]:
y_result = final_model.predict(X_result)
df_result = pd.DataFrame(y_result, columns=['Solution'], index=list(range(1, 9001)), dtype=np.int64)
df_result.to_csv('Result.csv', index_label='id')